<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/02_Data_Preprocessing_and_Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# 02.0 GOOGLE DRIVE + PROJECT SETUP
# ============================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import hashlib
import json
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

RAW_DIR = (
    PROJECT_ROOT /
    "data" /
    "raw"
)

PROCESSED_DIR = (
    PROJECT_ROOT /
    "data" /
    "processed"
)

RESULTS_DIR = (
    PROJECT_ROOT /
    "results"
)

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("AIR-LLM | NOTEBOOK 02")
print("=" * 90)
print(
    f"Project root : {PROJECT_ROOT}"
)
print(
    f"Raw data     : {RAW_DIR}"
)
print("=" * 90)

Mounted at /content/drive
AIR-LLM | NOTEBOOK 02
Project root : /content/drive/MyDrive/AIR_LLM_Research
Raw data     : /content/drive/MyDrive/AIR_LLM_Research/data/raw


In [2]:
# ============================================================
# 02.1 LOAD RAW DATASETS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

RAW_DIR = (
    PROJECT_ROOT /
    "data" /
    "raw"
)

SPLIT_DIR = (
    PROJECT_ROOT /
    "data" /
    "splits"
)

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

DATASET_FILES = {
    "adult_income":
        RAW_DIR / "adult_income.csv",

    "bank_marketing":
        RAW_DIR / "bank_marketing.csv",

    "diabetes_130us":
        RAW_DIR / "diabetes_130us.csv",
}

# ------------------------------------------------------------
# Adult Income
# ------------------------------------------------------------

ADULT_COLUMNS = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education_num",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "hours_per_week",
    "native_country",
    "income",
]


def load_adult_income(path):

    df = pd.read_csv(
        path,
        header=None,
        names=ADULT_COLUMNS,
        skipinitialspace=True,
        low_memory=False
    )

    return df


# ------------------------------------------------------------
# Bank Marketing
# ------------------------------------------------------------

def load_bank_marketing(path):

    df = pd.read_csv(
        path,
        sep=";",
        low_memory=False
    )

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    return df


# ------------------------------------------------------------
# Diabetes
# ------------------------------------------------------------

def load_diabetes(path):

    df = pd.read_csv(
        path,
        low_memory=False
    )

    df.columns = [
        str(c).strip()
        for c in df.columns
    ]

    return df


# ------------------------------------------------------------
# Load
# ------------------------------------------------------------

DATASETS_RAW = {}

for dataset_id in DATASET_IDS:

    path = DATASET_FILES[
        dataset_id
    ]

    if not path.exists():

        raise FileNotFoundError(
            f"Raw dataset not found:\n{path}"
        )

    if dataset_id == "adult_income":

        df = load_adult_income(path)

    elif dataset_id == "bank_marketing":

        df = load_bank_marketing(path)

    elif dataset_id == "diabetes_130us":

        df = load_diabetes(path)

    DATASETS_RAW[
        dataset_id
    ] = df.copy()

    target = TARGET_REGISTRY[
        dataset_id
    ]

    if target not in df.columns:

        raise KeyError(
            f"Target '{target}' missing from "
            f"{dataset_id}.\n"
            f"Columns: {list(df.columns)}"
        )

print("=" * 90)
print("RAW DATASETS LOADED")
print("=" * 90)

for dataset_id, df in DATASETS_RAW.items():

    print(
        f"{dataset_id:<20}"
        f"{len(df):>10,} rows | "
        f"{df.shape[1]:>3} columns | "
        f"target={TARGET_REGISTRY[dataset_id]}"
    )

RAW DATASETS LOADED
adult_income            32,561 rows |  15 columns | target=income
bank_marketing          45,211 rows |  17 columns | target=y
diabetes_130us         101,766 rows |  48 columns | target=readmitted


In [3]:
# ============================================================
# 02.2 REMOVE DUPLICATES
# ============================================================

DATASETS_DEDUPLICATED = {}

DUPLICATE_AUDIT = []

for dataset_id, df in DATASETS_RAW.items():

    rows_before = len(df)

    duplicate_count = int(
        df.duplicated().sum()
    )

    df_clean = (
        df
        .drop_duplicates(
            keep="first"
        )
        .reset_index(
            drop=True
        )
    )

    rows_after = len(df_clean)

    DATASETS_DEDUPLICATED[
        dataset_id
    ] = df_clean

    DUPLICATE_AUDIT.append({

        "dataset_id":
            dataset_id,

        "rows_before":
            rows_before,

        "duplicate_rows":
            duplicate_count,

        "rows_after":
            rows_after,

        "duplicates_removed":
            duplicate_count,

        "duplicate_rate":
            duplicate_count /
            rows_before,
    })

DUPLICATE_AUDIT_DF = pd.DataFrame(
    DUPLICATE_AUDIT
)

display(
    DUPLICATE_AUDIT_DF
)

,dataset_id,rows_before,duplicate_rows,rows_after,duplicates_removed,duplicate_rate
0,adult_income,32561,24,32537,24,0.000737
1,bank_marketing,45211,0,45211,0,0.000000
2,diabetes_130us,101766,0,101766,0,0.000000


In [4]:
# ============================================================
# 02.3 IDENTIFY NUMERICAL FEATURES
# ============================================================

NUMERICAL_FEATURES = {}

for dataset_id, df in (
    DATASETS_DEDUPLICATED.items()
):

    target = TARGET_REGISTRY[
        dataset_id
    ]

    features = df.drop(
        columns=[target]
    )

    numerical = (
        features
        .select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    NUMERICAL_FEATURES[
        dataset_id
    ] = numerical

for dataset_id, columns in (
    NUMERICAL_FEATURES.items()
):

    print(
        f"\n{dataset_id}"
    )

    print(
        f"Numerical features: "
        f"{len(columns)}"
    )

    print(columns)


adult_income
Numerical features: 6
['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

bank_marketing
Numerical features: 7
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

diabetes_130us
Numerical features: 11
['admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']


In [5]:
# ============================================================
# 02.4 IDENTIFY CATEGORICAL FEATURES
# ============================================================

CATEGORICAL_FEATURES = {}

for dataset_id, df in (
    DATASETS_DEDUPLICATED.items()
):

    target = TARGET_REGISTRY[
        dataset_id
    ]

    features = df.drop(
        columns=[target]
    )

    categorical = (
        features
        .select_dtypes(
            include=[
                "object",
                "string",
                "category"
            ]
        )
        .columns
        .tolist()
    )

    CATEGORICAL_FEATURES[
        dataset_id
    ] = categorical

for dataset_id, columns in (
    CATEGORICAL_FEATURES.items()
):

    print(
        f"\n{dataset_id}"
    )

    print(
        f"Categorical features: "
        f"{len(columns)}"
    )

    print(columns)


adult_income
Categorical features: 8
['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']

bank_marketing
Categorical features: 9
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

diabetes_130us
Categorical features: 36
['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


In [6]:
# ============================================================
# 02.5 TARGET SEPARATION
# ============================================================

FEATURES = {}
TARGETS = {}

for dataset_id, df in (
    DATASETS_DEDUPLICATED.items()
):

    target_column = TARGET_REGISTRY[
        dataset_id
    ]

    X = df.drop(
        columns=[target_column]
    ).copy()

    y = df[
        target_column
    ].copy()

    if len(X) != len(y):

        raise ValueError(
            f"Feature-target length mismatch "
            f"for {dataset_id}"
        )

    FEATURES[
        dataset_id
    ] = X

    TARGETS[
        dataset_id
    ] = y

print("=" * 90)
print("TARGET SEPARATION COMPLETED")
print("=" * 90)

for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id:<20}"
        f"X={FEATURES[dataset_id].shape} | "
        f"y={TARGETS[dataset_id].shape}"
    )

TARGET SEPARATION COMPLETED
adult_income        X=(32537, 14) | y=(32537,)
bank_marketing      X=(45211, 16) | y=(45211,)
diabetes_130us      X=(101766, 47) | y=(101766,)


In [7]:
# ============================================================
# 02.6 TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

TRAIN_FEATURES = {}
VALIDATION_FEATURES = {}
TEST_FEATURES = {}

TRAIN_TARGETS = {}
VALIDATION_TARGETS = {}
TEST_TARGETS = {}

SPLIT_INDICES = {}

for dataset_id in DATASET_IDS:

    X = FEATURES[
        dataset_id
    ]

    y = TARGETS[
        dataset_id
    ]

    # --------------------------------------------------------
    # First: train + temporary
    # --------------------------------------------------------

    X_train, X_temp, y_train, y_temp = (
        train_test_split(
            X,
            y,
            test_size=(
                TEST_SIZE +
                VALIDATION_SIZE
            ),
            random_state=RANDOM_STATE,
            shuffle=True,
            stratify=y
        )
    )

    # --------------------------------------------------------
    # Second: validation + test
    # --------------------------------------------------------

    relative_test_size = (
        TEST_SIZE /
        (
            TEST_SIZE +
            VALIDATION_SIZE
        )
    )

    X_val, X_test, y_val, y_test = (
        train_test_split(
            X_temp,
            y_temp,
            test_size=relative_test_size,
            random_state=RANDOM_STATE,
            shuffle=True,
            stratify=y_temp
        )
    )

    TRAIN_FEATURES[
        dataset_id
    ] = X_train.copy()

    VALIDATION_FEATURES[
        dataset_id
    ] = X_val.copy()

    TEST_FEATURES[
        dataset_id
    ] = X_test.copy()

    TRAIN_TARGETS[
        dataset_id
    ] = y_train.copy()

    VALIDATION_TARGETS[
        dataset_id
    ] = y_val.copy()

    TEST_TARGETS[
        dataset_id
    ] = y_test.copy()

    SPLIT_INDICES[
        dataset_id
    ] = {

        "train":
            X_train.index.tolist(),

        "validation":
            X_val.index.tolist(),

        "test":
            X_test.index.tolist(),
    }

print("=" * 90)
print("DATA SPLITTING COMPLETED")
print("=" * 90)

for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id:<20}"
        f"train={len(TRAIN_FEATURES[dataset_id]):>8,} | "
        f"val={len(VALIDATION_FEATURES[dataset_id]):>8,} | "
        f"test={len(TEST_FEATURES[dataset_id]):>8,}"
    )

DATA SPLITTING COMPLETED
adult_income        train=  19,522 | val=   6,507 | test=   6,508
bank_marketing      train=  27,126 | val=   9,042 | test=   9,043
diabetes_130us      train=  61,059 | val=  20,353 | test=  20,354


In [9]:
# ============================================================
# 02.7 FIT PREPROCESSING ON TRAINING DATA
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer

PREPROCESSORS = {}

for dataset_id in DATASET_IDS:

    numerical = NUMERICAL_FEATURES[
        dataset_id
    ]

    categorical = CATEGORICAL_FEATURES[
        dataset_id
    ]

    # --------------------------------------------------------
    # Numerical preprocessing
    # --------------------------------------------------------

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            ),
        ]
    )

    # --------------------------------------------------------
    # Categorical preprocessing
    # --------------------------------------------------------

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            ),
        ]
    )

    # --------------------------------------------------------
    # Combined preprocessing
    # --------------------------------------------------------

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                numeric_pipeline,
                numerical
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical
            ),
        ],
        remainder="drop"
    )

    # --------------------------------------------------------
    # CRITICAL: FIT ONLY ON TRAINING DATA
    # --------------------------------------------------------

    preprocessor.fit(
        TRAIN_FEATURES[
            dataset_id
        ]
    )

    PREPROCESSORS[
        dataset_id
    ] = preprocessor

    print(
        f"[OK] {dataset_id}: "
        f"training-only preprocessing fitted"
    )

print("=" * 90)
print(
    "TRAINING-ONLY PREPROCESSING COMPLETE"
)
print("=" * 90)

[OK] adult_income: training-only preprocessing fitted
[OK] bank_marketing: training-only preprocessing fitted
[OK] diabetes_130us: training-only preprocessing fitted
TRAINING-ONLY PREPROCESSING COMPLETE


In [10]:
# ============================================================
# 02.8 TRANSFORM VALIDATION DATA
# ============================================================

VALIDATION_TRANSFORMED = {}

for dataset_id in DATASET_IDS:

    preprocessor = PREPROCESSORS[
        dataset_id
    ]

    X_val = VALIDATION_FEATURES[
        dataset_id
    ]

    VALIDATION_TRANSFORMED[
        dataset_id
    ] = preprocessor.transform(
        X_val
    )

print(
    "Validation datasets transformed "
    "using training-fitted preprocessors."
)

Validation datasets transformed using training-fitted preprocessors.


In [11]:
# ============================================================
# 02.9 TRANSFORM TEST DATA
# ============================================================

TEST_TRANSFORMED = {}

for dataset_id in DATASET_IDS:

    preprocessor = PREPROCESSORS[
        dataset_id
    ]

    X_test = TEST_FEATURES[
        dataset_id
    ]

    TEST_TRANSFORMED[
        dataset_id
    ] = preprocessor.transform(
        X_test
    )

print(
    "Test datasets transformed "
    "using training-fitted preprocessors."
)

Test datasets transformed using training-fitted preprocessors.


In [12]:
# ============================================================
# 02.10 VERIFY NO LEAKAGE
# ============================================================

LEAKAGE_AUDIT = []

for dataset_id in DATASET_IDS:

    train_idx = set(
        TRAIN_FEATURES[
            dataset_id
        ].index
    )

    val_idx = set(
        VALIDATION_FEATURES[
            dataset_id
        ].index
    )

    test_idx = set(
        TEST_FEATURES[
            dataset_id
        ].index
    )

    train_val_overlap = (
        train_idx &
        val_idx
    )

    train_test_overlap = (
        train_idx &
        test_idx
    )

    val_test_overlap = (
        val_idx &
        test_idx
    )

    # --------------------------------------------------------
    # Target leakage
    # --------------------------------------------------------

    target = TARGET_REGISTRY[
        dataset_id
    ]

    assert (
        target not in
        TRAIN_FEATURES[
            dataset_id
        ].columns
    )

    assert (
        target not in
        VALIDATION_FEATURES[
            dataset_id
        ].columns
    )

    assert (
        target not in
        TEST_FEATURES[
            dataset_id
        ].columns
    )

    # --------------------------------------------------------
    # Split leakage
    # --------------------------------------------------------

    assert not train_val_overlap
    assert not train_test_overlap
    assert not val_test_overlap

    LEAKAGE_AUDIT.append({

        "dataset_id":
            dataset_id,

        "train_validation_overlap":
            len(train_val_overlap),

        "train_test_overlap":
            len(train_test_overlap),

        "validation_test_overlap":
            len(val_test_overlap),

        "target_leakage":
            False,

        "status":
            "PASSED",
    })

LEAKAGE_AUDIT_DF = pd.DataFrame(
    LEAKAGE_AUDIT
)

display(
    LEAKAGE_AUDIT_DF
)

,dataset_id,train_validation_overlap,train_test_overlap,validation_test_overlap,target_leakage,status
0,adult_income,0,0,0,False,PASSED
1,bank_marketing,0,0,0,False,PASSED
2,diabetes_130us,0,0,0,False,PASSED


In [13]:
# ============================================================
# 02.11 SAVE PROCESSED SPLITS
# ============================================================

for dataset_id in DATASET_IDS:

    dataset_dir = (
        SPLIT_DIR /
        dataset_id
    )

    dataset_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Raw-feature splits
    # --------------------------------------------------------

    TRAIN_FEATURES[
        dataset_id
    ].to_csv(
        dataset_dir /
        "X_train.csv",
        index=False
    )

    VALIDATION_FEATURES[
        dataset_id
    ].to_csv(
        dataset_dir /
        "X_validation.csv",
        index=False
    )

    TEST_FEATURES[
        dataset_id
    ].to_csv(
        dataset_dir /
        "X_test.csv",
        index=False
    )

    # --------------------------------------------------------
    # Target splits
    # --------------------------------------------------------

    TRAIN_TARGETS[
        dataset_id
    ].to_csv(
        dataset_dir /
        "y_train.csv",
        index=False
    )

    VALIDATION_TARGETS[
        dataset_id
    ].to_csv(
        dataset_dir /
        "y_validation.csv",
        index=False
    )

    TEST_TARGETS[
        dataset_id
    ].to_csv(
        dataset_dir /
        "y_test.csv",
        index=False
    )

    # --------------------------------------------------------
    # Preprocessed numerical arrays
    # --------------------------------------------------------

    np.save(
        dataset_dir /
        "X_validation_transformed.npy",
        VALIDATION_TRANSFORMED[
            dataset_id
        ]
    )

    np.save(
        dataset_dir /
        "X_test_transformed.npy",
        TEST_TRANSFORMED[
            dataset_id
        ]
    )

print("=" * 90)
print("PROCESSED SPLITS SAVED")
print("=" * 90)

for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id:<20}"
        f"{SPLIT_DIR / dataset_id}"
    )

PROCESSED SPLITS SAVED
adult_income        /content/drive/MyDrive/AIR_LLM_Research/data/splits/adult_income
bank_marketing      /content/drive/MyDrive/AIR_LLM_Research/data/splits/bank_marketing
diabetes_130us      /content/drive/MyDrive/AIR_LLM_Research/data/splits/diabetes_130us
